# This version has our baseline model with only H1 strategy

In [1]:
import pandas as pd
import pandas_ta as ta
import lightgbm as lgb
import numpy as np

# Loading the Data

In [2]:
df = pd.read_csv('btc_1hour_cleaned.csv')

## Setting the correct formats and cleaning Dataframe

In [3]:
df['Open time'] = pd.to_datetime(df['Open time'])
df['Close time'] = pd.to_datetime(df['Close time'])

In [4]:
df = df.set_index('Open time')

# Feature Engineering

## Functions for features: Strategy 1 + regime

In [5]:
# ROC
def compute_roc(df):
    df['roc_10'] = df['Close'].pct_change(periods=10)
    df['roc_21'] = df['Close'].pct_change(periods=21)
    return df

# MACD Histogram
def compute_macd(df):
    macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
    df['macd_histogram'] = macd['MACDh_12_26_9']
    return df

# ADX - Used as a feature and for regime detection
def compute_adx(df):
    df['adx'] = ta.adx(df['High'], df['Low'], df['Close'], length=14)['ADX_14']
    return df

# SMA 50 and SMA 200 is purely used for regime detection but not in feature matrix
def compute_regime(df):
    df['sma_50']  = df['Close'].rolling(50).mean()
    df['sma_200'] = df['Close'].rolling(200).mean()
    df['regime']  = (df['sma_50'] > df['sma_200']).astype(int).replace(0, -1)
    return df

# ATR 14
def compute_atr(df):
    df['atr_14'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    return df

# NATR
def compute_natr(df):
    df['natr'] = ta.natr(df['High'], df['Low'], df['Close'], length=14)
    return df

## Pipeline

In [6]:
def feature_pipeline(df):
    steps = [
        # model features (go to X)
        compute_roc,
        compute_macd,
        compute_adx,
        compute_atr,
        compute_natr,
        # execution layer only (do not go to X)
        compute_regime
    ]
    for step in steps:
        df = step(df)
    return df

In [7]:
df = feature_pipeline(df)

# Creating labels (y)

In [8]:
# Lookahead window - change this value to test different horizons
N = 5

# Step 1 - Forward return
df['forward_return'] = df['Close'].pct_change(periods=N).shift(-N)

In [9]:
# Step 2 - Binary Label
df['label'] = (df['forward_return'] > 0).astype(int)
# 1 - BUY, 0 - SELL

# Creating the lags

In [10]:
# Number of lags to create
N_lags = 5

feature_cols = ['Volume','roc_10', 'roc_21', 'macd_histogram', 'adx']

# Create lags for all feature columns
for col in feature_cols:
    for lag in range(1, N_lags + 1):
        df[f'{col}_lag{lag}'] = df[col].shift(lag)

# Drop NaN rows created by the lags
df = df.dropna(subset=feature_cols + [f'{col}_lag{i}' for col in feature_cols for i in range(1, N_lags + 1)] + ['label'])

# Exporting the Dataframe with all features and y as a Picklefile

In [11]:
# Only run this when we need to generate the dataframe to be used in our final model
# df.to_pickle('preprocessed.pkl')

In [12]:
# create a function to call this 
pd.read_pickle('../raw_data/preprocessed.pkl').head(3)

,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,...,macd_histogram_lag1,macd_histogram_lag2,macd_histogram_lag3,macd_histogram_lag4,macd_histogram_lag5,adx_lag1,adx_lag2,adx_lag3,adx_lag4,adx_lag5
Open time,,,,,,,,,,,,,,,,,,,,,
2018-01-02 14:00:00,13864.95,13894.86,13630.00,13664.40,705.180727,2018-01-02 14:59:59.999,9.692833e+06,7636,401.895566,5.524012e+06,...,37.305909,12.554494,10.095730,5.079898,-10.588927,43.104911,46.203100,48.414651,50.690652,53.143925
2018-01-02 15:00:00,13661.21,13739.89,13510.00,13690.03,952.656508,2018-01-02 15:59:59.999,1.298704e+07,6816,571.441429,7.790917e+06,...,37.855228,37.305909,12.554494,10.095730,5.079898,40.316812,43.104911,46.203100,48.414651,50.690652
2018-01-02 16:00:00,13690.02,13842.88,13584.99,13590.02,1020.856756,2018-01-02 16:59:59.999,1.404114e+07,7866,594.777166,8.182148e+06,...,37.556259,37.855228,37.305909,12.554494,10.095730,37.717466,40.316812,43.104911,46.203100,48.414651


# Creating the Feature Matrix and Y

In [13]:
X = df[['Volume','roc_10', 'roc_21', 'macd_histogram', 'adx',
                     'Volume_lag1', 'Volume_lag2', 'Volume_lag3', 'Volume_lag4', 'Volume_lag5',
                     'roc_10_lag1', 'roc_10_lag2', 'roc_10_lag3', 'roc_10_lag4', 'roc_10_lag5',
                     'roc_21_lag1', 'roc_21_lag2', 'roc_21_lag3', 'roc_21_lag4', 'roc_21_lag5',
                     'macd_histogram_lag1', 'macd_histogram_lag2', 'macd_histogram_lag3', 'macd_histogram_lag4', 'macd_histogram_lag5',
                     'adx_lag1', 'adx_lag2', 'adx_lag3', 'adx_lag4', 'adx_lag5']]

In [14]:
y = df[['label']]

# Training the Model

## Creating the Train / Test split according to user Input

In [16]:
# Hardcoding the date entered by user, to be set up dynamically in next versions
CUTOFF_DATE = '2025-01-01'

# Creating the training set for X and y
X_train = X[X.index < CUTOFF_DATE]
y_train = y[y.index < CUTOFF_DATE]

# Creating the testting set for X and y
X_predict = X[X.index >= CUTOFF_DATE]
y_predict = y[y.index >= CUTOFF_DATE]  # kept for evaluation only

print(f'Training on {len(X_train)} bars up to {CUTOFF_DATE}')
print(f'Predicting on {len(X_predict)} bars from {CUTOFF_DATE} onwards')

Training on 61208 bars up to 2025-01-01
Predicting on 10342 bars from 2025-01-01 onwards


## Training the model with LightGBM

In [17]:
# Class imbalance correction
scale_pos_weight = float((y_train['label'] == 0).sum() / (y_train['label'] == 1).sum())

# Settting the parametrs needed for the model
params = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'boosting_type':     'gbdt',
    'learning_rate':     0.05,
    'num_leaves':        31,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'scale_pos_weight':  scale_pos_weight,
    'seed':              42,
    'verbose':           -1
}

In [18]:
# Packaging the training data into LightGBM's required format
train_set_lgb = lgb.Dataset(X_train, label=y_train)

# Training the model
final_model= lgb.train(
    params= params,
    train_set= train_set_lgb,
    num_boost_round= 300,
    callbacks= [lgb.log_evaluation(period=-1)]
)

# Generate predictions on post-cutoff data
pred_proba_final = final_model.predict(X_predict)

## Building the signals DataFrame - this is the input to the execution filter

In [19]:
signals_df = df.loc[X_predict.index, ['Close', 'sma_50', 'sma_200', 'adx', 'atr_14']].copy()
signals_df['pred_proba'] = pred_proba_final
signals_df['true_label'] = y_predict.values

# Execution Filter + P&L Tracking

Purpose: Convert raw model probabilities into trade actions bar by bar

Why: Model output alone is not a trade — regime + confidence must confirm

MVP: Long only, bull regime only, confidence threshold filter

Entry metrics (stop distance, position size) are LOCKED at entry bar and never change during the trade — this prevents ATR drift from distorting stop levels and P&L calculations

Expandable: Add short trading in V2 by extending the bear regime block

Exit strategies used:
   1. Stop Loss     — price hits entry_price - trade_stop_dist (hard risk control)
   2. Regime Exit   — market structure no longer supports the trade
   3. Signal Exit   — model confidence drops below (1 - threshold)

In [20]:
# Setting the parameters of the execution strategy
CONFIDENCE_THRESHOLD = 0.55
INITIAL_CAPITAL = 1000.0
ATR_MULTIPLIER = 1.5
RISK_PCT = 0.01   # risk 1% of capital per trade
COST_PCT = 0.001  # 0.1% transaction cost per side

In [21]:
# Setting the loop that executes the trades

# State variables - maintained by the loop, not stored in Dataframe
position = 'flat'
entry_price = 0.0
trade_pos_size  = 0.0    # locked at entry, never changes mid-trade
trade_stop_dist = 0.0    # locked at entry, never changes mid-trade
capital = INITIAL_CAPITAL

# Storing the trade logs
trade_log = []

for timestamp, row in signals_df.iterrows():
    close   = row['Close']
    sma_50  = row['sma_50']
    sma_200 = row['sma_200']
    adx     = row['adx']
    atr     = row['atr_14']
    proba   = row['pred_proba']

    # Skip bars where indicators are not yet available (NaN warmup period)
    if pd.isna(atr) or pd.isna(adx) or pd.isna(sma_50) or pd.isna(sma_200):
        continue

    # Regime determination — checked every bar regardless of position
    # Bull requires price above both SMAs AND a strong trend (ADX > 25)
    # ADX < 25 = ranging/choppy market — signals unreliable
    bull = (close > sma_200) and (close > sma_50) and (adx > 25)

    # MANAGEMENT BLOCK — only runs when already in a trade (position == 'long')
    # Checks exit conditions in priority order:
    # Stop Loss → Regime Exit → Signal Exit

    if position == 'long':

        # EXIT 1: Stop Loss
        # Hard floor — exits immediately if price falls to locked stop level
        # stop_price is fixed at entry and never moves with ATR
        # This is the primary capital protection mechanism

        stop_price = entry_price - trade_stop_dist

        if close <= stop_price:
            pnl = (close - entry_price) * trade_pos_size
            cost = close * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp,
                'action':    'stop_loss',
                'close':     close,
                'pnl':       pnl,
                'capital':   capital
            })
            # Reset all trade state — trade_stop_dist also reset to avoid stale values
            position, entry_price, trade_pos_size, trade_stop_dist = 'flat', 0.0, 0.0, 0.0
            continue

        # EXIT 2: Regime Exit
        # If market structure shifts out of bull regime, close the long
        # regardless of model signal — regime gate overrides model
        # MVP is long-only so no point holding in bear/sideways

        if not bull:
            pnl = (close - entry_price) * trade_pos_size
            cost = close * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp,
                'action':    'close_long_regime',
                'close':     close,
                'pnl':       pnl,
                'capital':   capital
            })
            # Reset all trade state — trade_stop_dist also reset to avoid stale values
            position, entry_price, trade_pos_size, trade_stop_dist = 'flat', 0.0, 0.0, 0.0
            continue

        # EXIT 3: Signal Exit
        # Model confidence has flipped to strong SELL territory
        # Only exits if probability drops below (1 - threshold)
        # Probabilities near 0.5 do NOT trigger exit — only strong SELL signal does

        if proba <= (1-CONFIDENCE_THRESHOLD):
            pnl     = (close - entry_price) * trade_pos_size
            cost    = close * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp,
                'action':    'close_long_signal',
                'close':     close,
                'pnl':       pnl,
                'capital':   capital
            })
            position, entry_price, trade_pos_size, trade_stop_dist = 'flat', 0.0, 0.0, 0.0
            continue

    # ENTRY BLOCK - only runs when flat (position == 'flat')
    # All three conditions must be true simultaneously:
    #   1. Bull regime confirmed (regime gate)
    #   2. Model confidence above threshold (signal quality filter)
    # Entry metrics locked here — never recalculated during the trade

    else:
        if bull and proba >= CONFIDENCE_THRESHOLD:

            # Lock stop distance and position size at entry bar ATR
            # These values are frozen for the entire duration of this trade
            trade_stop_dist = ATR_MULTIPLIER * atr
            trade_pos_size = (capital * RISK_PCT) / trade_stop_dist

            cost = close * trade_pos_size * COST_PCT
            capital -= cost
            position = 'long'
            entry_price = close

            trade_log.append({
                'timestamp':  timestamp,
                'action':     'enter_long',
                'close':      close,
                'pnl':        0,
                'capital':    capital
            })

## Performance Summary

In [28]:
import numpy as np

# ================================================================
# FULL PERFORMANCE EVALUATION
# ================================================================

# --- Setup ---
trade_df       = pd.DataFrame(trade_log)
entries        = trade_df[trade_df['action'] == 'enter_long']
exits          = trade_df[trade_df['action'] != 'enter_long']
winning_trades = exits[exits['pnl'] > 0]
losing_trades  = exits[exits['pnl'] < 0]
total_closed   = len(exits)
win_rate       = len(winning_trades) / total_closed * 100 if total_closed > 0 else 0
loss_rate      = len(losing_trades)  / total_closed * 100 if total_closed > 0 else 0

# --- Risk Metrics ---
trade_df_sorted = trade_df.sort_values('timestamp')
trade_df_sorted['capital_return'] = trade_df_sorted['capital'].pct_change()
mean_return      = trade_df_sorted['capital_return'].mean()
std_return       = trade_df_sorted['capital_return'].std()
sharpe           = (mean_return / std_return) * np.sqrt(8760)
trade_df_sorted['cummax']   = trade_df_sorted['capital'].cummax()
trade_df_sorted['drawdown'] = (trade_df_sorted['capital'] - trade_df_sorted['cummax']) / trade_df_sorted['cummax']
max_drawdown     = trade_df_sorted['drawdown'].min() * 100
gross_profit     = winning_trades['pnl'].sum()
gross_loss       = losing_trades['pnl'].abs().sum()
profit_factor    = gross_profit / gross_loss if gross_loss > 0 else 0
days_in_backtest = (trade_df_sorted['timestamp'].iloc[-1] - trade_df_sorted['timestamp'].iloc[0]).days
annualised_return = ((capital / INITIAL_CAPITAL) ** (365 / days_in_backtest) - 1) * 100

# --- Buy and Hold Benchmark ---
buy_date    = CUTOFF_DATE
buy_price   = df.loc[buy_date:, 'Close'].iloc[0]
final_price = df['Close'].iloc[-1]
btc_units   = INITIAL_CAPITAL / buy_price
bnh_value   = btc_units * final_price
bnh_return  = ((bnh_value - INITIAL_CAPITAL) / INITIAL_CAPITAL) * 100

# --- Transaction Costs ---
implied_costs = trade_df['pnl'].sum() - (capital - INITIAL_CAPITAL)

# --- Print All Results ---
print(f'╔══════════════════════════════════════╗')
print(f'║         BACKTEST RESULTS             ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ CAPITAL                              ║')
print(f'║  Initial:         ${INITIAL_CAPITAL:>10,.2f}        ║')
print(f'║  Final:           ${capital:>10,.2f}        ║')
print(f'║  Total Return:    {((capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100):>9.2f}%         ║')
print(f'║  Annualised:      {annualised_return:>9.2f}%         ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ TRADE STATISTICS                     ║')
print(f'║  Total Trades:    {len(entries):>10}         ║')
print(f'║  Winning Trades:  {len(winning_trades):>10}         ║')
print(f'║  Losing Trades:   {len(losing_trades):>10}         ║')
print(f'║  Win Rate:        {win_rate:>9.1f}%         ║')
print(f'║  Loss Rate:       {loss_rate:>9.1f}%         ║')
print(f'║  Avg Win PnL:     ${winning_trades["pnl"].mean():>10.2f}        ║')
print(f'║  Avg Loss PnL:    ${losing_trades["pnl"].mean():>10.2f}        ║')
print(f'║  Implied Costs:   ${implied_costs:>10.2f}        ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ RISK METRICS                         ║')
print(f'║  Sharpe Ratio:    {sharpe:>10.2f}         ║')
print(f'║  Max Drawdown:    {max_drawdown:>9.2f}%         ║')
print(f'║  Profit Factor:   {profit_factor:>10.2f}         ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ BUY & HOLD BENCHMARK                 ║')
print(f'║  Buy Price:       ${buy_price:>10,.2f}        ║')
print(f'║  Final Price:     ${final_price:>10,.2f}        ║')
print(f'║  Final Value:     ${bnh_value:>10,.2f}        ║')
print(f'║  B&H Return:      {bnh_return:>9.2f}%         ║')
print(f'║  Strategy Return: {((capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100):>9.2f}%         ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ ACTION BREAKDOWN                     ║')
for action, count in trade_df['action'].value_counts().items():
    print(f'║  {action:<20} {count:>6}         ║')
print(f'╚══════════════════════════════════════╝')

╔══════════════════════════════════════╗
║         BACKTEST RESULTS             ║
╠══════════════════════════════════════╣
║ CAPITAL                              ║
║  Initial:         $  1,000.00        ║
║  Final:           $    777.00        ║
║  Total Return:       -22.30%         ║
║  Annualised:         -19.44%         ║
╠══════════════════════════════════════╣
║ TRADE STATISTICS                     ║
║  Total Trades:           123         ║
║  Winning Trades:          65         ║
║  Losing Trades:           58         ║
║  Win Rate:             52.8%         ║
║  Loss Rate:            47.2%         ║
║  Avg Win PnL:     $      6.55        ║
║  Avg Loss PnL:    $     -6.86        ║
║  Implied Costs:   $    251.17        ║
╠══════════════════════════════════════╣
║ RISK METRICS                         ║
║  Sharpe Ratio:        -13.37         ║
║  Max Drawdown:       -23.49%         ║
║  Profit Factor:         1.07         ║
╠══════════════════════════════════════╣
║ BUY & HOLD BEN